# Google Batch Runs


## Setup


In [ ]:
# Libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Pipeline

%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

# Pipeline Check


needs = {'run': ['write_batch', 'read_batch', 'batch_path', 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path',
                   'model_slug', 'make_directories'],
         'backends': ['USAGE', 'spent', 'record_usage', 'takes_sampling'],
         'settings': ['BATCHES_DIR', 'MODELS', 'GENERATION']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

# Signature Check

import inspect
returns = len(inspect.getsource(run.read_batch).rsplit('return ', 1)[1].split(','))
if returns != 5:
    raise SystemExit(f'run.read_batch returns {returns} values, this notebook '
                     f'expects 5.\nCopy scripts/ from the latest package and '
                     f'restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

## Model


In [ ]:
MODEL = 'gemini-3.5-flash-lite'

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
if spec['provider'] != 'google':
    raise SystemExit(f'{MODEL} is served by {spec["provider"]}, not Google')

prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print(f'Model      {MODEL}')
print(f'Billed at  ${spec["price"]["input"]}/M input, '
      f'${spec["price"]["output"]}/M output including thinking, halved on a batch')
print(f'Sampling   temperature {settings.GENERATION["temperature"]}, '
      f'top_p {settings.GENERATION["top_p"]}')
print(f'Cap        {settings.GENERATION["max_tokens"]} tokens')
print(f'Collected  {have:,} of {wanted:,}')
print(f'Key found  {bool(utils.api_key("google"))}')

## Rerun


In [ ]:
FRESH = False        # Fresh Run

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

## Requests


In [ ]:
path, count = run.write_batch(MODEL)

if path is None:
    print('Nothing outstanding for this model')
else:
    print(f'{count:,} requests written to {path}')
    print()
    print(json.dumps(json.loads(path.read_text().splitlines()[0]), indent=2))

## Cost


In [ ]:
OUTPUT_TOKENS = 400          # Output Tokens
INPUT_TOKENS = 24

if path is None:
    print('Nothing to price, this model is already collected')
else:
    price = spec['price']
    standard = (count * INPUT_TOKENS * price['input']
                + count * OUTPUT_TOKENS * price['output']) / 1e6
    print(f'{count:,} calls at {OUTPUT_TOKENS} output tokens each')
    print(f'  Standard  ${standard:,.2f}')
    print(f'  Batched   ${standard / 2:,.2f}')

## Submit


In [ ]:
if path is None:
    raise SystemExit(
        'Nothing outstanding for this model, so there is nothing to submit.\n'
        'To re-read a batch you already have, run from the repository root:\n'
        '    python scripts/run.py ingest --model <model> --file <output>.jsonl')

from google import genai
from google.genai import types

client = genai.Client(api_key=utils.api_key('google'))

uploaded = client.files.upload(
    file=str(path),
    config=types.UploadFileConfig(display_name=path.stem, mime_type='jsonl'))
job = client.batches.create(model=MODEL, src=uploaded.name,
                            config={'display_name': f'{MODEL}-adaptation'})

job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job_file.parent.mkdir(parents=True, exist_ok=True)
job_file.write_text(job.name)
print(f'Submitted {job.name}, {job.state.name}')

print(f'Requests kept at {run.name_after_job(MODEL, job.name.split("/")[-1])}')

## Resume Job


In [ ]:
from google import genai

client = genai.Client(api_key=utils.api_key('google'))

jobs = [{'name': b.name, 'state': b.state.name,
         'display_name': getattr(b, 'display_name', ''),
         'created': str(getattr(b, 'create_time', ''))}
        for b in client.batches.list(config={'page_size': 10})]
display(pd.DataFrame(jobs))

In [ ]:
# Select Job
ADOPT = jobs[0]['name'] if 'jobs' in dir() and jobs else ''

if ADOPT:
    job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
    job_file.parent.mkdir(parents=True, exist_ok=True)
    job_file.write_text(ADOPT)
    print(f'Adopted {ADOPT} for {MODEL}')

## Status


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')

if not job_file.exists():
    print('No job submitted for this model yet, run the cell above')
else:
    job = client.batches.get(name=job_file.read_text().strip())
    print(f'Job {job.name}')
    print(f'{job.state.name}')
    if job.state.name == 'JOB_STATE_FAILED':
        print(f'Error: {getattr(job, "error", "no detail given")}')

## Results


In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job = (client.batches.get(name=job_file.read_text().strip())
       if job_file.exists() else None)

if job is None or job.state.name != 'JOB_STATE_SUCCEEDED':
    print(f'Nothing to read yet: {job.state.name if job else "no job adopted"}')
else:
    results = run.batch_path(MODEL, 'output', job.name.split('/')[-1])
    results.write_bytes(client.files.download(file=job.dest.file_name))
    print(f'Downloaded {results}')

    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, results)

    usage, cost = backends.USAGE, backends.spent(MODEL)
    print(f'\nRead {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output '
          f'including thinking')
    print(f'Cost: ${cost:,.2f} standard, ${cost / 2:,.2f} batched')
    print(f'Output tokens a reply: '
          f'{usage["output"] / max(read - failed, 1):.0f}')

## Validation


In [ ]:
# Update Flags

import flags
flags.apply(MODEL)
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    marked = lambda name: collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['True', 'true']) \
        | ~collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['', 'False', 'false', 'nan'])
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored, '
          f'{int(marked("blocked").sum())} blocked, '
          f'{int(marked("truncated").sum())} truncated')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))